In [1]:
import os

In [2]:
os.chdir("../")
%pwd


'e:\\Projects\\Kidney Disease Classification'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [4]:
from CNNClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from CNNClassifier.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
    ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )

        return data_ingestion_config

In [6]:
import os
import zipfile
import gdown
from CNNClassifier import logger
from CNNClassifier.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion", exist_ok=True)
            logger.info(f"Downloading file from {dataset_url} to {zip_download_dir}")
            file_id = dataset_url.split('/')[-2]
            preffix = "https://drive.google.com/uc?/export=download&id="
            gdown.download(preffix + file_id, str(zip_download_dir))
            logger.info(f"Downloaded file size: {get_size(zip_download_dir)}")
        except Exception as e:
            logger.exception(f"Error occurred while downloading the file: {e}")
            raise e

    def extract_zip_files(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
            logger.info(f"Unzipped {self.config.local_data_file} to {unzip_path}")



In [10]:
try:    
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_files()
except Exception as e:
    logger.exception(f"Error occurred in data ingestion: {e}")
    raise e

[2026-09-20 00:36:06,435: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-09-20 00:36:06,437: INFO: common: yaml file: params.yaml loaded successfully]
[2026-09-20 00:36:06,438: INFO: common: created directory at: artifacts]
[2026-09-20 00:36:06,439: INFO: common: created directory at: artifacts/data_ingestion]
[2026-09-20 00:36:06,439: INFO: 671858878: Downloading file from https://drive.google.com/file/d/1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3/view?usp=sharing to artifacts\data_ingestion\data.zip]


Downloading...
From (original): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3
From (redirected): https://drive.google.com/uc?id=1vlhZ5c7abUKF8xXERIw6m9Te8fW7ohw3&confirm=t&uuid=2b2d8403-3fce-4164-8827-9d8b31fe2314
To: e:\Projects\Kidney Disease Classification\artifacts\data_ingestion\data.zip
100%|██████████| 57.7M/57.7M [00:04<00:00, 14.4MB/s]

[2026-09-20 00:36:13,581: INFO: 671858878: Downloaded file size: ~ 56361 KB]


[2026-09-20 00:36:13,958: INFO: 671858878: Unzipped artifacts\data_ingestion\data.zip to artifacts\data_ingestion]
